# 📖 第四课：过拟合与正则化

**目标**：理解过拟合的本质，掌握正则化技术，学会在偏差与方差之间找到平衡

---

## 📚 学习目标
- 理解欠拟合、刚好拟合、过拟合三种状态
- 掌握偏差-方差权衡的核心思想
- 学会使用 L1 (Lasso) 和 L2 (Ridge) 正则化
- 理解正则化强度参数的影响
- 学会用学习曲线诊断模型问题

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.font_manager as fm
import pathlib
_cache = matplotlib.get_cachedir()
for _f in pathlib.Path(_cache).glob('fontlist*.json'):
    _f.unlink(missing_ok=True)
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.sans-serif'] = ['PingFang HK', 'PingFang SC', 'Heiti TC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import learning_curve, train_test_split
import warnings
warnings.filterwarnings('ignore')

print("库导入成功！")

---

## 1. 过拟合现象可视化

### 三种拟合状态
- **欠拟合**：模型太简单，没学会规律
- **刚好拟合**：模型学到了真实规律
- **过拟合**：模型记住了噪声，泛化差

In [ ]:
# 生成带噪声的非线性数据
np.random.seed(42)
X = np.linspace(-3, 3, 60).reshape(-1, 1)
y_true = np.sin(X).ravel()
y = y_true + np.random.normal(0, 0.2, X.shape[0])

# 分割训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"训练集: {len(X_train)} 样本")
print(f"测试集: {len(X_test)} 样本")

In [ ]:
# 三种拟合状态对比
models = {
    '欠拟合 (1次)': make_pipeline(PolynomialFeatures(1), LinearRegression()),
    '刚好拟合 (3次)': make_pipeline(PolynomialFeatures(3), LinearRegression()),
    '过拟合 (15次)': make_pipeline(PolynomialFeatures(15), LinearRegression())
}

X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    y_plot = model.predict(X_plot)
    
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    
    ax.scatter(X_train, y_train, s=30, alpha=0.6, c='blue', label='训练数据')
    ax.plot(X_plot, y_true, 'g--', linewidth=1, alpha=0.5, label='真实函数')
    ax.plot(X_plot, y_plot, 'r-', linewidth=2, label='模型拟合')
    ax.set_title(f'{name}\n训练 R²={train_score:.3f}, 测试 R²={test_score:.3f}')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('欠拟合 vs 刚好拟合 vs 过拟合', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 观察要点:")
print("  - 欠拟合：训练和测试得分都很低")
print("  - 过拟合：训练得分很高，但测试得分低")
print("  - 刚好拟合：训练和测试得分都较高且接近")

---

## 2. 偏差-方差权衡

In [ ]:
# 用不同多项式次数展示偏差-方差权衡
degrees = range(1, 20)
train_scores = []
test_scores = []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(degrees, train_scores, 'b-o', label='训练 R²', markersize=5)
plt.plot(degrees, test_scores, 'r-s', label='测试 R²', markersize=5)
plt.axvline(x=3, color='green', linestyle='--', alpha=0.7, label='最佳复杂度')
plt.fill_between(degrees, train_scores, test_scores, alpha=0.1, color='red')
plt.xlabel('模型复杂度（多项式次数）')
plt.ylabel('R² 分数')
plt.title('偏差-方差权衡')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 关键洞察:")
print("  - 模型太简单 → 高偏差（欠拟合）")
print("  - 模型太复杂 → 高方差（过拟合）")
print("  - 最佳模型 → 偏差和方差的平衡点")

---

## 3. L2 正则化（Ridge 岭回归）

在损失函数中加入权重平方和的惩罚项：

$$\text{Loss} = \text{原始损失} + \alpha \sum w_i^2$$

- $\alpha$ 越大，正则化越强，权重越趋向于 0
- 不会产生稀疏解，所有特征都保留

In [ ]:
# Ridge 正则化对比
alphas = [0, 0.001, 0.01, 0.1, 1, 10]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, alpha in enumerate(alphas):
    model = make_pipeline(PolynomialFeatures(15), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    y_plot = model.predict(X_plot)
    
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    
    axes[i].scatter(X_train, y_train, s=20, alpha=0.5, c='blue')
    axes[i].plot(X_plot, y_true, 'g--', linewidth=1, alpha=0.5)
    axes[i].plot(X_plot, y_plot, 'r-', linewidth=2)
    axes[i].set_title(f'α={alpha}\n训练 R²={train_score:.3f}, 测试 R²={test_score:.3f}')
    axes[i].set_ylim(-2, 2)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('L2 正则化 (Ridge) — 不同强度对比', fontsize=14)
plt.tight_layout()
plt.show()

---

## 4. L1 正则化（Lasso 回归）

在损失函数中加入权重绝对值的惩罚项：

$$\text{Loss} = \text{原始损失} + \alpha \sum |w_i|$$

- 特点：能将某些权重压缩为 0，实现**自动特征选择**
- 产生稀疏解

In [ ]:
# Lasso 正则化 — 观察权重稀疏性
from sklearn.datasets import make_regression

# 生成多特征数据（其中一些特征是噪声）
X_multi, y_multi = make_regression(n_samples=100, n_features=20, n_informative=5, 
                                     noise=10, random_state=42)

# 对比 Ridge 和 Lasso 的权重
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=1.0)

ridge.fit(X_multi, y_multi)
lasso.fit(X_multi, y_multi)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge 权重
axes[0].bar(range(20), ridge.coef_, color='steelblue')
axes[0].set_title(f'Ridge (L2) — 非零权重: {np.sum(ridge.coef_ != 0)}')
axes[0].set_xlabel('特征编号')
axes[0].set_ylabel('权重值')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Lasso 权重
axes[1].bar(range(20), lasso.coef_, color='coral')
axes[1].set_title(f'Lasso (L1) — 非零权重: {np.sum(lasso.coef_ != 0)}')
axes[1].set_xlabel('特征编号')
axes[1].set_ylabel('权重值')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('L2 vs L1 正则化的权重对比', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nRidge 非零权重数: {np.sum(ridge.coef_ != 0)}/{len(ridge.coef_)}")
print(f"Lasso 非零权重数: {np.sum(lasso.coef_ != 0)}/{len(lasso.coef_)}")
print("\n💡 Lasso 自动将不重要特征的权重置为 0（特征选择）")

---

## 5. 学习曲线诊断

In [ ]:
# 用学习曲线判断欠拟合 vs 过拟合
from sklearn.linear_model import LinearRegression

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    (PolynomialFeatures(1), '欠拟合 (1次)'),
    (PolynomialFeatures(3), '刚好拟合 (3次)'),
    (PolynomialFeatures(15), '过拟合 (15次)')
]

for ax, (poly, title) in zip(axes, configs):
    model = make_pipeline(poly, LinearRegression())
    
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='neg_mean_squared_error'
    )
    
    train_mean = -train_scores.mean(axis=1)
    val_mean = -val_scores.mean(axis=1)
    
    ax.plot(train_sizes, train_mean, 'b-o', label='训练误差', markersize=4)
    ax.plot(train_sizes, val_mean, 'r-s', label='验证误差', markersize=4)
    ax.fill_between(train_sizes, train_mean - train_scores.std(axis=1),
                     train_mean + train_scores.std(axis=1), alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_scores.std(axis=1),
                     val_mean + val_scores.std(axis=1), alpha=0.1, color='red')
    ax.set_title(title)
    ax.set_xlabel('训练样本数')
    ax.set_ylabel('均方误差')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('学习曲线诊断', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 诊断方法:")
print("  - 欠拟合：训练误差和验证误差都很高且接近")
print("  - 过拟合：训练误差很低，验证误差很高，差距大")
print("  - 刚好拟合：两条曲线收敛到较低误差")

---

## 6. 正则化实战：选择最佳 α

In [ ]:
from sklearn.model_selection import cross_val_score

# 用交叉验证选择最佳正则化强度
alphas = np.logspace(-4, 3, 20)
ridge_scores = []
lasso_scores = []

for alpha in alphas:
    ridge = make_pipeline(PolynomialFeatures(10), Ridge(alpha=alpha))
    lasso = make_pipeline(PolynomialFeatures(10), Lasso(alpha=alpha, max_iter=10000))
    
    ridge_scores.append(cross_val_score(ridge, X, y, cv=5).mean())
    lasso_scores.append(cross_val_score(lasso, X, y, cv=5).mean())

plt.figure(figsize=(10, 6))
plt.semilogx(alphas, ridge_scores, 'b-o', label='Ridge (L2)', markersize=5)
plt.semilogx(alphas, lasso_scores, 'r-s', label='Lasso (L1)', markersize=5)
plt.xlabel('正则化强度 α (log scale)')
plt.ylabel('交叉验证 R²')
plt.title('选择最佳正则化强度')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_ridge_alpha = alphas[np.argmax(ridge_scores)]
best_lasso_alpha = alphas[np.argmax(lasso_scores)]
print(f"Ridge 最佳 α: {best_ridge_alpha:.4f} (R²={max(ridge_scores):.3f})")
print(f"Lasso 最佳 α: {best_lasso_alpha:.4f} (R²={max(lasso_scores):.3f})")

---

## 📝 课后思考

1. 如果模型在训练集和测试集上表现都很差，应该增大还是减小正则化强度？
2. L1 正则化为什么能产生稀疏解？从几何角度思考。
3. 在深度学习中，除了 L1/L2 还有哪些防止过拟合的方法？

---

**下一步**：决策树基础——学习最直观的机器学习算法，用 if-else 规则构建树形分类器。